# L7.4 · SmolVLA: adentro del VLM

| Notebook | Qué abre |
|---|---|
| L7.1 | El mapa: qué entra, qué sale, dónde están los 450 M de parámetros |
| L7.2 | La imagen: de la foto a 64 tokens |
| L7.3 | La fila del VLM: palabras, estado y quién puede mirar a quién |
| **L7.4** | Adentro del VLM: qué miran las palabras, por qué 16 de 32 capas, qué queda guardado |
| `06` (flujo) | Entrenarlo con nuestros datos |
| L7.5 | El experto de acciones: la brújula de L6 con ojos |
| L7.6 | Las 10 pasadas: un bloque de verdad pasando de ruido a movimiento |
| L7.7 | El entrenamiento por dentro |
| `07` (flujo) | Evaluación contra ACT, en los mismos episodios |

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import time

import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import DATASETS_DIR
from toolkit.policy import smolvla_internals as vla
from toolkit.policy.smolvla_g1_base import G1_BASE_DIR, load_g1_policy

plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN = DATASETS_DIR / "local" / "pick_mameluco_act_v1_train"
train_meta = LeRobotDataset("local/pick_mameluco_act_v1_train", root=TRAIN).meta
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=DATASETS_DIR / "local" / "pick_mameluco_act_v1")
# El VLM queda congelado al entrenar: el modelo base y el entrenado lo tienen idéntico.
policy, preprocess, _ = load_g1_policy(G1_BASE_DIR, train_meta.stats, device)
model = policy.model
tokenizer = model.vlm_with_expert.processor.tokenizer
text_model = model.vlm_with_expert.get_vlm_model().text_model

frame = dataset[int(dataset.meta.episodes["dataset_from_index"][10]) + 120]  # el instante de L7.1
obs = vla.prepare_observation(policy, preprocess(dict(frame)))
with torch.no_grad():
    prefix = vla.embed_prefix(model, obs)
image = obs.images[0]


def show(ax, img):
    ax.imshow(((img[0].permute(1, 2, 0).float().cpu() + 1) / 2).clamp(0, 1).numpy())
    ax.axis("off")

## 1. El problema

La fila de L7.3 tiene 113 tokens, pero cada uno sabe sólo de sí mismo: el
token de "shelf" no sabe dónde está el estante en esta foto, y el del estado no
sabe qué hay adelante del robot. Para que cada token se cargue del contexto,
la fila pasa por **capas**: en cada una, cada token mira a los otros (los que
la máscara le deja) y se actualiza con lo que encontró. SmolVLA usa las
primeras 16 de las 32 capas del VLM de texto.

## 2. Una capa por dentro

Cada capa hace dos cosas, siempre en el mismo orden:

1. **Atención**: cada token arma una *pregunta* (Q), una *etiqueta* (K) y un
   *contenido* (V). Compara su pregunta con las etiquetas de los demás, y se
   lleva una mezcla de sus contenidos, pesada por cuánto coinciden.
2. **MLP**: cada token, solo, procesa lo que juntó.

Antes de cada paso, una normalización (RMSNorm) lleva cada token a un largo
parecido, y después de cada paso el resultado se **suma** al token original
(así nada se pierde de capa en capa). Las matrices de una capa:

In [ ]:
layer = text_model.layers[0]
rows = []
for name, parameter in layer.named_parameters():
    rows.append((name, tuple(parameter.shape[::-1]), parameter.numel()))
for name, shape, count in rows:
    print(f"  {name:<36} {str(shape):>14}  {count / 1e6:5.2f} M")
per_layer = sum(c for *_, c in rows)
print(f"  {'total por capa':<36} {'':>14}  {per_layer / 1e6:5.2f} M   × 16 capas = {16 * per_layer / 1e6:.0f} M")
heads, kv_heads = model.vlm_with_expert.num_attention_heads, model.vlm_with_expert.num_key_value_heads
print(f"\ncabezas de atención: {heads} preguntas, {kv_heads} etiquetas/contenidos, de {layer.self_attn.head_dim} números cada una")

Casi 10 M de parámetros por capa, y dos tercios están en el MLP. Las
etiquetas y los contenidos (K, V) son más chicos que las preguntas (320 contra
960): las 15 cabezas de preguntas comparten 5 de etiquetas y contenidos, de a
3 (se llama GQA). Eso importa en la sección 5: lo que se guarda para las
pasadas es K y V, y así ocupa un tercio.

## 3. Qué mira cada palabra en la imagen, capa por capa

Capturamos la atención real mientras la fila pasa por las 16 capas. Antes de
mirar zona por zona, una sorpresa: sumamos cuánta atención recibe cada uno de
los 64 tokens de imagen desde las palabras, en todas las capas.

In [ ]:
with torch.no_grad(), vla.capture_attention(model) as log:
    kv_cache = vla.run_prefix(model, prefix)
attention = torch.stack([p[0].mean(0) for p in log.probs])  # (16 capas, 113, 113), promedio de las cabezas
print("atención capturada:", tuple(attention.shape))

text_start = prefix.segments["texto"].start
pieces = [p.replace("Ġ", "") for p in tokenizer.convert_ids_to_tokens(obs.lang_tokens[0].tolist())]
queries = {f"'{pieces[i]}'": text_start + i for i in [3, 4, 8]}  # folded, cover, shelf
queries["estado"] = prefix.segments["estado"].start
layers = [0, 5, 10, 15]

words_slice = slice(text_start, text_start + int(obs.lang_masks[0].sum()))
received = attention[:, words_slice, :64].mean(dim=(0, 1))
with torch.no_grad():
    lengths = vla.image_tokens(model, image)[0].float().norm(dim=1).cpu()
sink = int(received.argmax())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
show(axes[0], image)
axes[0].imshow(np.kron(received.reshape(8, 8).numpy(), np.ones((64, 64))), cmap="inferno", alpha=0.6, extent=(0, 512, 512, 0))
axes[0].set_title("atención que recibe cada token de imagen\n(desde las palabras, todas las capas)")
axes[1].scatter(lengths, received, s=15)
axes[1].scatter(lengths[sink], received[sink], s=60, color="red")
axes[1].annotate(f"token {sink}", (lengths[sink], received[sink]), textcoords="offset points", xytext=(-50, -5))
axes[1].set_xlabel("largo del vector del token")
axes[1].set_ylabel("atención que recibe")
plt.tight_layout()
print(f"el token {sink} recibe {received[sink] / received.mean():.1f} veces el promedio, "
      f"y su vector mide {lengths[sink] / lengths.median():.1f} veces la mediana")


Un token se lleva mucha más atención que el resto: el de la esquina de
abajo a la derecha. No es porque ahí haya algo importante. Probamos con otros
instantes de otros episodios (escenas distintas) y siempre gana el mismo
token, y su vector es mucho más largo que el de los demás. Es lo que se llama
un **sumidero de atención**: un lugar donde las cabezas descargan la atención
cuando no tienen nada útil que mirar (la suma tiene que dar 1 y la atención
"sobrante" va a parar ahí). Pasa en muchos transformers, de texto y de
imágenes.

Ahora sí, zona por zona. Para cada token elegido, cuánta de su atención va a
cada token de imagen (promedio de las 15 cabezas), pintado sobre la foto. Cada
mapa tiene su propia escala, calculada **sin el sumidero** (marcado con ×):
muestra **dónde** mira, no cuánto.

In [ ]:
fig, axes = plt.subplots(len(queries), len(layers), figsize=(14, 3.6 * len(queries)))
sink_xy = ((sink % 8) * 64 + 32, (sink // 8) * 64 + 32)
for row, (name, q) in enumerate(queries.items()):
    for col, layer_index in enumerate(layers):
        ax = axes[row, col]
        to_image = attention[layer_index, q, :64].clone()
        share = to_image.sum().item()
        to_image[sink] = float("nan")
        to_image = (to_image / np.nanmax(to_image.numpy())).reshape(8, 8).numpy()
        show(ax, image)
        ax.imshow(np.kron(to_image, np.ones((64, 64))), cmap="inferno", alpha=0.6, extent=(0, 512, 512, 0))
        ax.plot(*sink_xy, marker="x", color="white", markersize=10)
        ax.set_title(f"{name} · capa {layer_index} · {share:.0%} a la imagen", fontsize=9)
plt.tight_layout()

Para mirar:

- En la **capa 0**, las palabras reparten mucha atención por la imagen (casi
  60 %), bastante pareja: todavía no saben qué buscar.
- En las **capas del medio** casi no la miran (menos de 15 %): están armando el
  sentido de la frase entre palabras.
- En las **últimas** vuelven a la imagen (20 a 40 %).
- Hay alguna zona reconocible ("cover", en la capa 10, mira el borde de arriba
  del paquete), pero también se repite la esquina de arriba a la derecha, que
  es relleno negro: otro sumidero, más chico (fue el segundo token más mirado
  en los tres instantes que probamos).
- Son mapas ruidosos: promedian 15 cabezas que miran cosas distintas, y SigLIP
  no se entrenó para describir zonas finas (L7.2). No hay que leerlos como
  "la palabra encontró el objeto".

¿Y a dónde va el resto de la atención? Repartimos la atención de cada token
entre la imagen (separando la parte de relleno negro), las palabras y él
mismo, en cada capa:

In [ ]:
image_rows_padding = 24  # los tokens de imagen que ven sólo relleno (L7.2)
real_words = slice(text_start, text_start + int(obs.lang_masks[0].sum()))


def split_attention(q):
    a = attention[:, q]
    return {
        "imagen: relleno": a[:, :image_rows_padding].sum(-1),
        "imagen: foto": a[:, image_rows_padding:64].sum(-1),
        "palabras": a[:, real_words].sum(-1) - (a[:, q] if real_words.start <= q < real_words.stop else 0),
        "sí mismo": a[:, q],
    }


fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
colors = ["#444444", "tab:blue", "tab:green", "tab:orange"]
for ax, (name, q) in zip(axes, [("'shelf'", queries["'shelf'"]), ("estado", queries["estado"])]):
    bottom = torch.zeros(16)
    for (part, values), color in zip(split_attention(q).items(), colors):
        ax.bar(range(16), values, bottom=bottom, color=color, label=part)
        bottom += values
    ax.set_xlabel("capa")
    ax.set_title(f"a dónde mira {name}")
axes[0].set_ylabel("fracción de la atención")
axes[1].legend(fontsize=8, loc="lower right")
plt.tight_layout()

Las palabras se miran sobre todo entre ellas, y la imagen se lleva una parte
en la primera capa y en las últimas. El token del estado, en cambio, casi no
mira la imagen: después de la capa 0 se dedica a las palabras y a sí mismo.

Ojo con eso último. La matriz que arma el token del estado **se entrena**, y
acá todavía es la de `smolvla_base`, hecha para brazos SO-100. La imagen y las
palabras no miran al estado (L7.3), así que lo que hacen es idéntico antes y
después de entrenar; lo que hace el token del estado, no. Lo volvemos a mirar
con el modelo entrenado en L7.5.

## 4. ¿Por qué 16 capas de 32?

Más capas cuestan más, y el robot espera. El paper midió qué pasa si el experto
lee las capas de sólo las primeras 8, 16, 24 o las 32, en el benchmark de
simulación LIBERO (tabla 8; éxito promedio):

In [ ]:
paper = {"8 capas": 75.0, "16 capas": 78.5, "24 capas": 79.5, "32 capas": 80.3,
         "32 salteando\nde a 2": 75.5, "VLM más chico\n(256 M)": 75.8}
fig, ax = plt.subplots(figsize=(9, 3.2))
bars = ax.bar(list(paper), list(paper.values()), color=["lightgray", "tab:green", "lightgray", "lightgray", "lightgray", "lightgray"])
for bar, value in zip(bars, paper.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.3, f"{value}", ha="center")
ax.set_ylim(70, 83)
ax.set_ylabel("éxito promedio (%)")
ax.set_title("SmolVLA en LIBERO según las capas del VLM (paper, tabla 8)")
plt.tight_layout()

Con 16 capas se pierden 1,8 puntos contra usar las 32, a mitad de costo.
Saltear capas (usar las 32 de a una sí y una no) o usar un VLM más chico sale
peor que quedarse con las primeras 16. Por eso SmolVLA tira la mitad de arriba
del VLM antes de empezar.

## 5. Qué queda guardado para las 10 pasadas

En cada capa, cada token de la fila dejó su etiqueta (K) y su contenido (V).
Eso es lo único que el experto de acciones va a consultar del VLM: no
necesita la fila entera, sólo poder preguntarle cosas. Se guarda una vez (la
**caché de K y V**) y se reusa en las 10 pasadas.

In [ ]:
first = kv_cache[0]
print("capas guardadas:", len(kv_cache))
print("por capa · K:", tuple(first["key_states"].shape), "· V:", tuple(first["value_states"].shape),
      "  (lote, tokens de la fila, cabezas, números por cabeza)")
numbers = sum(v["key_states"].numel() + v["value_states"].numel() for v in kv_cache.values())
print(f"total guardado: {numbers / 1e6:.2f} M números ({numbers * 2 / 2**20:.1f} MB en bfloat16)")


def timed(fn, repeats=20):
    for _ in range(3):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    if device == "cuda":
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - start) / repeats


noise = torch.randn(1, 50, 32, device=device)
t1 = torch.ones(1, device=device)
with torch.no_grad():
    prefix_ms = timed(lambda: vla.run_prefix(model, vla.embed_prefix(model, obs)))
    step_ms = timed(lambda: vla.velocity(model, prefix, kv_cache, noise, t1))
print(f"la fila por el VLM (una vez): {prefix_ms:.0f} ms · una pasada del experto: {step_ms:.0f} ms")
print(f"un bloque = 1 × {prefix_ms:.0f} + 10 × {step_ms:.0f} ≈ {prefix_ms + 10 * step_ms:.0f} ms "
      f"(sin guardar, serían ≈ {10 * (prefix_ms + step_ms):.0f} ms)")

La fila cuesta lo que varias pasadas del experto (los números exactos
dependen de la máquina y de qué más esté usando la GPU). Sin la caché, cada
una de las 10 pasadas tendría que volver a pasar la fila por el VLM. Y lo que
se guarda es poco: unos 2 MB por instante.

## Lo que nos llevamos

- Cada capa: atención (cada token pregunta y se lleva una mezcla de lo que
  encontró) y MLP (cada token procesa solo). 15 cabezas de preguntas comparten
  5 de etiquetas y contenidos.
- Las palabras de la instrucción miran zonas de la imagen, y el estado mira la
  escena: es la forma en que el contexto entra a cada token.
- Usar 16 capas de 32 cuesta poco en éxito y ahorra la mitad.
- De todo esto, al experto le queda la caché de K y V de cada capa, que se
  calcula una vez y se consulta en las 10 pasadas.

## Referencias

- Shukor et al., [SmolVLA](https://arxiv.org/abs/2506.01844) (2025), tabla 8.
- Xiao et al., [Attention Sinks](https://arxiv.org/abs/2309.17453) (2023) y Darcet et al.,
  [Vision Transformers Need Registers](https://arxiv.org/abs/2309.16588) (2023): los sumideros.
- Ainslie et al., [GQA](https://arxiv.org/abs/2305.13245) (2023): varias
  cabezas de preguntas compartiendo etiquetas y contenidos.
- Su et al., [RoPE](https://arxiv.org/abs/2104.09864) (2021): cómo entra la
  posición, girando las preguntas y etiquetas.